# Regression Trees, Random Forest and Gradient Boosting – Fuel Consumption of New Cars

**Dataset:** Fuel consumption of 701 new car models of model year 2025 (Natural Resources Canada, Open Government
Licence – Canada): make, model, vehicle class, engine size, cylinders, transmission, fuel type and fuel consumption.
File: `../00_Data/car_fuel_consumption_2025.csv`.

**Business question:** How much fuel (L/100 km) does a new car model use, depending on its engine, vehicle class,
transmission and fuel type? The features interact (a hybrid car with a continuously variable transmission uses little
fuel almost independently of its engine size, a conventional car uses much more with a larger engine) – a typical use
case for tree-based models.

**After this exercise you can ...**
- find the best split of a regression tree by hand (SSE criterion),
- fit, visualise and prune a regression tree,
- explain and apply bagging, random forests (incl. out-of-bag error, permutation importance) and gradient boosting,
- compare models fairly with cross-validation.

## Concept

The concepts behind this exercise (formulas and worked examples) are
explained in the notebook [regression_trees_ensembles_concept.ipynb](regression_trees_ensembles_concept.ipynb). Read it before you start.

## Libraries and settings

In [ ]:
# Libraries
import os
import platform
import warnings
from datetime import datetime
from platform import python_version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import tree
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    KFold,
)
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

# Ignore warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

## Exercise 1: Calculations by hand

*Use the notebook cell below as your calculator: type the formulas with the numbers in
Python (e.g. `(3 + 5) / 2` or `0.4 ** 2`) – no paper needed.*

**a) A split of a regression tree.** Six cars with the following engine sizes and fuel consumptions:

| engine size $x$ [litres] | 1.5 | 2.0 | 2.5 | 3.5 | 5.0 | 6.2 |
|:---|:---|:---|:---|:---|:---|:---|
| fuel consumption $y$ [L/100 km] | 6.5 | 7.5 | 8.5 | 11.0 | 13.0 | 14.0 |

The tree considers the split *engine size ≤ 3 litres*: left node {6.5, 7.5, 8.5}, right node {11.0, 13.0, 14.0}.
Compute the SSE of the root node (all six cars) and of both child nodes (the function `sse()` is given). By how much
does the split reduce the SSE? What are the leaf predictions (means of the two nodes)?

**b) Variance of an average.** 100 trees have variance $\sigma^2 = 4$ each. Compute the variance of their average
$\rho\sigma^2 + \frac{1-\rho}{B}\sigma^2$ for $\rho = 0.6$ (bagged trees, similar to each other) and for $\rho = 0.2$
(random forest).

**c) One boosting step.** The current prediction for a car is 9.0 L/100 km, the observed consumption is 11.0. The next
tree predicts the residual exactly. What is the new prediction with learning rate $\nu = 0.1$?

In [ ]:
# a) Fuel consumption (L/100 km) of six cars, sorted by engine size
y = np.array([6.5, 7.5, 8.5, 11.0, 13.0, 14.0])
left = np.array([6.5, 7.5, 8.5])  # engine size <= 3 litres
right = np.array([11.0, 13.0, 14.0])  # engine size > 3 litres


# Function: computes the sum of squared deviations of v from its mean
def sse(v):
    """Sum of squared errors around the mean."""
    return np.sum((v - v.mean()) ** 2)


# SSE of the root and of both children, leaf predictions (means)
# Your code here

# b) Variance of the average of B = 100 trees with variance 4
# Your code here

# c) One boosting step: prediction 9.0, observed 11.0, learning rate 0.1
# Your code here

---

> ✏️ **Your answer**
>
> *Double-click this cell and write your answer here.*

---

## Exercise 2: Import and explore the car data

a) Import the car data. The target is the column `combined` (fuel consumption in L/100 km, combined city and
highway driving). Use the features `engine_size`, `cylinders`, `vehicle_class`, `transmission_type` and `fuel_type`
(the text columns as dummy variables: `pd.get_dummies(..., columns=[...], dtype=int)`).

b) Plot `combined` against `engine_size`, separately for cars with a continuously variable transmission and for all
other cars. What do you see?

c) Split the data into training (80 %) and test set (20 %) with `random_state=42`.

In [ ]:
# a) Import the data as cars
#    File: '../00_Data/car_fuel_consumption_2025.csv'
# Your code here

# Features (text columns as dummy variables: one 0/1 column per category)
features = [
    'engine_size',
    'cylinders',
    'vehicle_class',
    'transmission_type',
    'fuel_type',
]
X = ___
y = cars['combined']

# b) Scatter plot for cars with and without continuously variable transmission
# Your code here

# c) Train/test split
X_train, X_test, y_train, y_test = ___

---

> ✏️ **Your answer**
>
> *Double-click this cell and write your answer here.*

---

## Exercise 3: Baseline – linear regression

Fit a multiple linear regression with all features on the training set and compute RMSE and $R^2$ on the test set. This
is the baseline that the tree-based models have to beat.

In [ ]:
# Exercise 3: linear regression with all features; test RMSE and R²
# Your code here

## Exercise 4: A small regression tree

a) Fit a `DecisionTreeRegressor(max_depth=3, random_state=42)` on the training set.

b) Visualise the tree with `tree.plot_tree()` and interpret the first two levels. Which features are used?

c) Compute RMSE and $R^2$ on the test set. How does the small tree compare to the linear regression?

In [ ]:
# a) Fit the tree
reg_tree = ___

# b) Plot the tree
# Your code here

# c) Test RMSE and R²
# Your code here

---

> ✏️ **Your answer**
>
> *Double-click this cell and write your answer here.*

---

## Exercise 5: Tree complexity and pruning

a) Fit trees with `max_depth` = 1, 2, 4, 6, 8, 10 and 15. For each depth, compute the RMSE on the training set and the
5-fold CV RMSE (on the training set). Plot both errors against the depth. From which depth on does a deeper tree no
longer help?

b) Instead of limiting the depth, you can grow a full tree and **prune** it. The parameter `ccp_alpha` controls how
strongly: the larger `ccp_alpha`, the smaller the tree. For `ccp_alpha` = 0, 0.005, 0.01, 0.02, 0.05 and 0.1, compute the
5-fold CV RMSE and the number of leaves (`model.get_n_leaves()`). Choose the value with the smallest CV RMSE, store it as
`best_alpha` and compute the test RMSE of this pruned tree.

In [ ]:
# Exercise 5: tree complexity and pruning
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# a) Training error and CV error for a few depths
depths = [1, 2, 4, 6, 8, 10, 15]
rmse_train = []
rmse_cv = []
for d in depths:
    model = DecisionTreeRegressor(max_depth=d, random_state=42)
    # Your code here: 5-fold CV RMSE and training RMSE (append both)

# Plot of both errors against the depth
# Your code here

# b) CV RMSE and number of leaves for a few values of ccp_alpha
for alpha in [0, 0.005, 0.01, 0.02, 0.05, 0.1]:
    model = DecisionTreeRegressor(ccp_alpha=alpha, random_state=42)
    # Your code here

# Best value (smallest CV RMSE) as best_alpha; test RMSE of the pruned tree
# Your code here

---

> ✏️ **Your answer**
>
> *Double-click this cell and write your answer here.*

---

## Exercise 6: Random forest

a) Fit random forests with `n_estimators` = 1, 5, 10, 25, 50, 100, 200, 400 (`oob_score=True`, `random_state=42`,
`n_jobs=-1`). The OOB predictions are stored in `oob_prediction_`. Plot the OOB RMSE against the number of trees. When do
the gains plateau?

b) Compare `max_features` = 1.0 (all features, i.e. bagging), `'sqrt'` and 0.5 with 5-fold CV (300 trees).

c) For the best forest, compute the test RMSE. Compare the impurity-based feature importance (`feature_importances_`)
with the **permutation importance** on the test set.

In [ ]:
# a) Number of trees vs. OOB error
n_trees = [1, 5, 10, 25, 50, 100, 200, 400]
oob_rmse = []
for n in n_trees:
    rf = RandomForestRegressor(
        n_estimators=n, oob_score=True, random_state=42, n_jobs=-1
    )
    rf.fit(X_train, y_train)
    # OOB prediction of each training car (NaN if the car was never OOB)
    oob_pred = rf.oob_prediction_
    ok = ~np.isnan(oob_pred)
    # Your code here

# Plot of the OOB RMSE
# Your code here

# b) 5-fold CV RMSE for different values of max_features
# Your code here

In [ ]:
# c) Random forest with the best setting from b): test RMSE and feature
#    importance (impurity-based and permutation importance)
# Your code here

---

> ✏️ **Your answer**
>
> *Double-click this cell and write your answer here.*

---

## Exercise 7: Gradient boosting

a) Fit `GradientBoostingRegressor(n_estimators=1000, max_depth=3, random_state=42)` with learning rates
$\nu$ = 0.01, 0.1 and 1.0. Use `staged_predict(X_test)` to compute the test RMSE after each boosting iteration and plot
the three curves.

*Note: we look at the test curve here only for illustration. In practice, `learning_rate` and `n_estimators` are tuned
with cross-validation (see Exercise 8).*

b) Describe the influence of the learning rate. What happens with $\nu = 1.0$?

In [ ]:
# a) Test error after each tree for three learning rates
for lr in [0.01, 0.1, 1.0]:
    gb = ___

    # staged_predict gives the predictions after 1, 2, ..., 1000 trees
    rmse_staged = []
    for y_pred in gb.staged_predict(X_test):
        rmse_staged.append(root_mean_squared_error(y_test, y_pred))
    plt.plot(range(1, 1001), rmse_staged, label='learning rate ' + str(lr))

# Figure showing the test error by number of trees
plt.ylim(1, 3)
plt.xlabel('Number of trees M')
plt.ylabel('Test RMSE [L/100 km]')
plt.legend()
plt.show()

---

> ✏️ **Your answer**
>
> *Double-click this cell and write your answer here.*

---

## Exercise 8: Fair model comparison

Compare the following models with **5-fold CV RMSE on the training set** and summarise the results in a table:
linear regression, pruned tree (best `ccp_alpha` from Exercise 5), random forest (best setting from Exercise 6) and
gradient boosting (`learning_rate=0.05`, `n_estimators=300`, `max_depth=3`). Then evaluate the best model **once** on the
test set. Which model would you recommend to the car importer, and what are the drawbacks?

In [ ]:
# Exercise 8: the four models (settings from Exercises 5 and 6)
models = {
    'Linear regression': LinearRegression(),
    'Pruned tree': ___,
    'Random forest': ___,
    'Gradient boosting': ___,
}

# 5-fold CV RMSE of all models (store them in cv_rmse)
cv_rmse = {}
for name, model in models.items():
    # Your code here
    pass

# Fit the best model and evaluate it once on the test set
# Your code here

---

> ✏️ **Your answer**
>
> *Double-click this cell and write your answer here.*

---

## Exercise 9: More covariates

So far the models used the engine, the vehicle class, the transmission and the fuel type. The data set contains more
information: the **make** of the car (39 makes, e.g. Toyota, BMW, Ford).

a) Add the make as dummy variables and split the data again (80/20, `random_state=42` – the same cars as before).

b) Compute the 5-fold CV RMSE of the linear regression, the random forest (best setting of Exercise 6) and gradient
boosting (as in Exercise 8) with the **extended** feature set and compare with Exercise 8.

c) Evaluate the random forest with the extended feature set on the test set and compute its permutation importance.
How important are the single makes?

In [ ]:
# a) Extended feature set: the make as additional dummy variables
features_more = features + ['make']
X_more = ___

# Same random_state as before -> the same training and test cars
X2_train, X2_test, y2_train, y2_test = ___

# b) CV RMSE with the extended feature set, compared with Exercise 8
# Your code here

# c) Random forest with the extended feature set: test RMSE and permutation
#    importance
# Your code here

---

> ✏️ **Your answer**
>
> *Double-click this cell and write your answer here.*

---

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Show operating system, date/time and Python version
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')